# Chapter 03: Generative AI Foundations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abda-dc/ai-engineering-zero-to-hero/blob/main/book/chapters/part2-core-genai/ch03-generative-ai-foundations.ipynb)

**Part:** Part II: Core Generative AI & Large Language Models  
**Chapter Identifier:** `ch03`  

## Core Concepts Covered in this Chapter:
- **GenAI vs. Discriminative**
- **Neural Nets**
- **Transformers**
- **Self-Attention**
- **Tokens**
- **Embeddings**
- **Foundation Models**
- **Pre-training/Fine-tuning**
- **Diffusion**
- **GANs/VAEs**
- **Multimodal**
- **RLHF**
- **Evals**
- **AI Safety**

---


## 1. Generative AI vs. Discriminative Models

Discriminative models learn the conditional probability P(Y|X) to classify or predict labels. Generative models learn the joint probability distribution P(X, Y) or data distribution P(X) to sample and generate entirely new synthetic data instances.

```mermaid
graph TD
    subgraph Discriminative
        D_in[Input X] --> D_model[Decision Boundary] --> D_out["P(Y|X): Label Y"]
    end
    subgraph Generative
        G_in[Latent Noise or Prompt] --> G_model[Learned Distribution P(X)] --> G_out["P(X): New Token / Image / Audio"]
    end
```


In [ ]:
def discriminative_classify(text: str) -> str:
    return "SPAM" if "free money" in text.lower() else "HAM"

def generative_sample(prompt: str) -> str:
    return prompt + " is a revolutionary field transforming software engineering."

print("Discriminative Output:", discriminative_classify("Win free money now"))
print("Generative Output:", generative_sample("Artificial Intelligence"))

## 2. Neural Networks, Self-Attention & Transformers

Transformers replaced sequential recurrence with Self-Attention, allowing every token in a sequence to attend to every other token simultaneously. Scaled Dot-Product Attention: Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V.

```mermaid
flowchart LR
    Q[Query Q] --> M[MatMul]
    K[Key K] --> M
    M --> S[Scale sqrt d_k]
    S --> Mask[Optional Causal Mask]
    Mask --> SM[Softmax]
    SM --> V_Mul[MatMul with Value V]
    V[Value V] --> V_Mul
    V_Mul --> Out[Attention Output]
```


In [ ]:
import math

def softmax(x: list) -> list:
    max_x = max(x)
    exp_x = [math.exp(i - max_x) for i in x]
    sum_exp = sum(exp_x)
    return [i / sum_exp for i in exp_x]

def scaled_dot_product_attention_1d(q: list, keys: list, values: list) -> tuple:
    d_k = len(q)
    scores = [sum(qi * ki for qi, ki in zip(q, k)) / math.sqrt(d_k) for k in keys]
    weights = softmax(scores)
    output_dim = len(values[0])
    output = [0.0] * output_dim
    for weight, val in zip(weights, values):
        for i in range(output_dim):
            output[i] += weight * val[i]
    return output, weights

q = [1.0, 0.5]
keys = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
values = [[0.9, 0.1], [0.2, 0.8], [0.7, 0.7]]

out, weights = scaled_dot_product_attention_1d(q, keys, values)
print("Attention Weights:", [round(w, 4) for w in weights])
print("Aggregated Value Representation:", [round(o, 4) for o in out])

## 3. The Modern Lifecycle: Pre-training, Fine-Tuning, RLHF & Safety

Building foundation models follows a 4-stage pipeline: (1) Pre-training on trillions of tokens, (2) Supervised Fine-Tuning (SFT) on instructions, (3) Reinforcement Learning from Human Feedback (RLHF / DPO) for alignment, and (4) Evals & Guardrails for AI Safety.


In [ ]:
pipeline_stages = [
    {"stage": "1. Self-Supervised Pre-Training", "objective": "Next-Token Prediction on Web Scale Text", "artifact": "Base Model"},
    {"stage": "2. Supervised Fine-Tuning (SFT)", "objective": "Instruction following (Q&A, Coding)", "artifact": "Instruct Model"},
    {"stage": "3. RLHF / DPO Alignment", "objective": "Helpfulness, Honesty, Harmlessness (HHH)", "artifact": "Aligned Model"},
    {"stage": "4. Evals & Safety Guardrails", "objective": "Jailbreak prevention, toxicity filters", "artifact": "Production Model"}
]

for s in pipeline_stages:
    print(f"[{s['stage']}]")
    print(f"  Goal: {s['objective']}")
    print(f"  Result: {s['artifact']}")
    print()

---
## Hands-On Capstone Lab: Build a Scaled Dot-Product Attention Engine with Causal Masking

### Objective:
Implement an autoregressive causal attention block where tokens are prevented from attending to future tokens (lookahead mask).

### Instructions:
1. Create a sequence of 3 tokens.
2. Compute attention score matrix Q * K^T.
3. Apply causal mask (-1e9 for future positions) before applying softmax.


In [ ]:
def causal_attention_matrix(Q: list, K: list) -> list:
    n = len(Q)
    d_k = len(Q[0])
    raw_matrix = []
    for i in range(n):
        row = []
        for j in range(n):
            if j > i:
                row.append(-1e9)
            else:
                score = sum(Q[i][d] * K[j][d] for d in range(d_k)) / math.sqrt(d_k)
                row.append(score)
        raw_matrix.append(softmax(row))
    return raw_matrix

Q_tokens = [[1.0, 0.2], [0.5, 0.8], [0.2, 1.0]]
K_tokens = [[1.0, 0.2], [0.5, 0.8], [0.2, 1.0]]

attn_matrix = causal_attention_matrix(Q_tokens, K_tokens)
print("Causal Attention Weight Matrix (Lower Triangular):")
for row_idx, row in enumerate(attn_matrix):
    print(f"Token {row_idx}: {[round(val, 4) for val in row]}")

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: Why is the dot product scaled by 1/sqrt(d_k) in Scaled Dot-Product Attention?
- **A)** To prevent gradients from vanishing when dot products grow large in high dimensions
- **B)** To speed up matrix multiplication by converting floats to integers
- **C)** To enforce causal lookahead masking
- **D)** To ensure tokens are converted to lowercase

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) To prevent gradients from vanishing when dot products grow large in high dimensions

**Explanation:** For large projection dimensions d_k, dot products grow large in magnitude, pushing softmax into regions with extremely small gradients. Dividing by sqrt(d_k) counteracts this effect.
</details>

### Question 2: What is the primary objective of Reinforcement Learning from Human Feedback (RLHF)?
- **A)** To teach the model basic grammar from raw unlabelled text
- **B)** To align the model's behavior with human preferences for helpfulness, honesty, and harmlessness
- **C)** To compress the model weights into 4-bit integers
- **D)** To speed up token generation during inference

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) To align the model's behavior with human preferences for helpfulness, honesty, and harmlessness

**Explanation:** RLHF fine-tunes an instruction-tuned model using a reward model trained on human preference rankings to align responses with human values and safety standards.
</details>

### Question 3: In autoregressive decoder-only LLMs (like GPT), what is the purpose of the causal attention mask?
- **A)** It hides punctuation from the model
- **B)** It prevents positions from attending to subsequent (future) tokens during next-token prediction
- **C)** It encrypts model weights to prevent reverse engineering
- **D)** It converts text into image tokens

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) It prevents positions from attending to subsequent (future) tokens during next-token prediction

**Explanation:** Causal masking sets attention scores for future positions (j > i) to negative infinity, ensuring that generating token t only depends on tokens at indices <= t.
</details>

### Question 4: What generative paradigm models data generation by learning to iteratively reverse a progressive noise addition process?
- **A)** Generative Adversarial Networks (GANs)
- **B)** Variational Autoencoders (VAEs)
- **C)** Diffusion Models
- **D)** Decision Trees

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** C) Diffusion Models

**Explanation:** Diffusion models learn the reverse of a Markovian forward process that gradually adds Gaussian noise to data.
</details>
